<a href="https://colab.research.google.com/github/Jtuaz/Projects-Portfolio-/blob/main/CNN_From_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import math
import numpy as np
import torch
import torchvision

# TRANSFORMS IMPLEMENTED FROM SCRATCH

class CUSTOM_TO_TENSOR:
    """Converts a raw PIL Image or NumPy array (H x W x C) in [0, 255] to a FloatTensor (C x H x W) in [0.0, 1.0]."""

    def __call__(self, INPUT_IMAGE):
        # Convert PIL Image to NumPy array if necessary (required for torchvision Datasets)
        if not isinstance(INPUT_IMAGE, np.ndarray):
            INPUT_IMAGE = np.array(INPUT_IMAGE)

        # Convert integer array (0 to 255) to float tensor and scale values to floating point range [0.0, 1.0]
        TENSOR_IMAGE = torch.from_numpy(INPUT_IMAGE).float() / 255.0

        # Rearrange tensor dimensions from Height-Width-Channel (H, W, C) to PyTorch's expected Channel-Height-Width (C, H, W)
        TRANSPOSED_TENSOR = TENSOR_IMAGE.permute(2, 0, 1)  # (H, W, C) -> (C, H, W)

        return TRANSPOSED_TENSOR


class CUSTOM_NORMALIZE:
    """Standardizes input tensor per channel using the formula: (X - MEAN) / STD."""

    def __init__(self, MEAN_TUPLE, STD_TUPLE):
        # Reshape 1D mean and std tuples into (3, 1, 1) tensors so broadcasting works across spatial dimensions (C, H, W)
        self.MEAN_VECTOR = torch.tensor(MEAN_TUPLE).view(3, 1, 1)
        self.STD_VECTOR = torch.tensor(STD_TUPLE).view(3, 1, 1)

    def __call__(self, INPUT_TENSOR):
        # Apply element-wise standard score formula across image channels using PyTorch broadcasting
        NORMALIZED_TENSOR = (INPUT_TENSOR - self.MEAN_VECTOR) / self.STD_VECTOR

        return NORMALIZED_TENSOR


class CUSTOM_COMPOSE_TRANSFORMS:
    """Sequentially chains multiple custom transforms together."""

    def __init__(self, TRANSFORM_LIST):
        # Store the list of transformation callables
        self.TRANSFORM_LIST = TRANSFORM_LIST

    def __call__(self, INPUT_DATA):
        PROCESSED_DATA = INPUT_DATA

        # Pass the input data sequentially through each transformation step in the list
        for INDIVIDUAL_TRANSFORM in self.TRANSFORM_LIST:
            PROCESSED_DATA = INDIVIDUAL_TRANSFORM(PROCESSED_DATA)

        return PROCESSED_DATA


# CUSTOM NEURAL NETWORK LAYERS FROM SCRATCH


class FROM_SCRATCH_CONVOLUTION_2D(torch.nn.Module):
    """2D Convolutional Layer implemented from scratch using matrix multiplication (im2col / unfold)."""

    def __init__(self, IN_CHANNELS, OUT_CHANNELS, KERNEL_SIZE=3, PADDING=1, STRIDE=1):
        super(FROM_SCRATCH_CONVOLUTION_2D, self).__init__()

        self.IN_CHANNELS = IN_CHANNELS
        self.OUT_CHANNELS = OUT_CHANNELS
        self.KERNEL_SIZE = KERNEL_SIZE
        self.PADDING = PADDING
        self.STRIDE = STRIDE

        # He / Kaiming Normal Initialization bounds to prevent vanishing/exploding gradients
        BOUND_VAL = 1.0 / math.sqrt(IN_CHANNELS * KERNEL_SIZE * KERNEL_SIZE)

        # Initialize trainable weight tensor: shape (OUT_CHANNELS, IN_CHANNELS, KERNEL_SIZE, KERNEL_SIZE)
        self.WEIGHT_TENSOR = torch.nn.Parameter(
            torch.randn(OUT_CHANNELS, IN_CHANNELS, KERNEL_SIZE, KERNEL_SIZE) * BOUND_VAL
        )

        # Initialize trainable bias vector: one bias parameter per output channel
        self.BIAS_VECTOR = torch.nn.Parameter(torch.zeros(OUT_CHANNELS))

    def forward(self, INPUT_TENSOR):
        BATCH_SIZE, CHANNELS, HEIGHT, WIDTH = INPUT_TENSOR.shape

        # Apply zero-padding manually to input image borders if padding > 0
        if self.PADDING > 0:
            PADDED_INPUT = torch.nn.functional.pad(
                INPUT_TENSOR, (self.PADDING, self.PADDING, self.PADDING, self.PADDING)
            )
        else:
            PADDED_INPUT = INPUT_TENSOR

        # Calculate spatial height and width of output feature maps
        OUTPUT_HEIGHT = (HEIGHT + 2 * self.PADDING - self.KERNEL_SIZE) // self.STRIDE + 1
        OUTPUT_WIDTH = (WIDTH + 2 * self.PADDING - self.KERNEL_SIZE) // self.STRIDE + 1

        # Extract sliding receptive field windows into flat column vectors (im2col approach)
        UNFOLDED_COLUMNS = torch.nn.functional.unfold(
            PADDED_INPUT, kernel_size=self.KERNEL_SIZE, stride=self.STRIDE
        )

        # Flatten 4D weight kernel tensor into a 2D matrix
        FLATTENED_WEIGHTS = self.WEIGHT_TENSOR.view(self.OUT_CHANNELS, -1)

        # Compute convolution via matrix multiplication
        CONVOLUTION_OUTPUT = torch.matmul(FLATTENED_WEIGHTS, UNFOLDED_COLUMNS)

        # Add channel biases using broadcasting shape (1, OUT_CHANNELS, 1)
        CONVOLUTION_OUTPUT += self.BIAS_VECTOR.view(1, -1, 1)

        # Reshape flat matrix output back to 4D spatial feature map
        FINAL_FEATURE_MAP = CONVOLUTION_OUTPUT.view(BATCH_SIZE, self.OUT_CHANNELS, OUTPUT_HEIGHT, OUTPUT_WIDTH)

        return FINAL_FEATURE_MAP


class FROM_SCRATCH_LINEAR(torch.nn.Module):
    """Fully Connected (Dense) Layer implemented from scratch: Y = X * W^T + B."""

    def __init__(self, IN_FEATURES, OUT_FEATURES):
        super(FROM_SCRATCH_LINEAR, self).__init__()

        BOUND_VAL = 1.0 / math.sqrt(IN_FEATURES)
        self.WEIGHT_MATRIX = torch.nn.Parameter(torch.randn(OUT_FEATURES, IN_FEATURES) * BOUND_VAL)
        self.BIAS_VECTOR = torch.nn.Parameter(torch.zeros(OUT_FEATURES))

    def forward(self, INPUT_TENSOR):
        DENSE_OUTPUT = torch.matmul(INPUT_TENSOR, self.WEIGHT_MATRIX.t()) + self.BIAS_VECTOR
        return DENSE_OUTPUT


class FROM_SCRATCH_RELU(torch.nn.Module):
    """Rectified Linear Unit Activation from scratch: f(x) = max(0, x)."""

    def forward(self, INPUT_TENSOR):
        return torch.clamp(INPUT_TENSOR, min=0.0)


class FROM_SCRATCH_MAX_POOL_2D(torch.nn.Module):
    """2D Max Pooling layer implemented from scratch by reshaping spatial grids."""

    def __init__(self, KERNEL_SIZE=2, STRIDE=2):
        super(FROM_SCRATCH_MAX_POOL_2D, self).__init__()
        self.KERNEL_SIZE = KERNEL_SIZE
        self.STRIDE = STRIDE

    def forward(self, INPUT_TENSOR):
        BATCH_SIZE, CHANNELS, HEIGHT, WIDTH = INPUT_TENSOR.shape

        OUTPUT_HEIGHT = HEIGHT // self.STRIDE
        OUTPUT_WIDTH = WIDTH // self.STRIDE

        # Reshape feature tensor to isolate pooling windows into independent tensor axes
        RESHAPED_TENSOR = INPUT_TENSOR.view(
            BATCH_SIZE, CHANNELS, OUTPUT_HEIGHT, self.KERNEL_SIZE, OUTPUT_WIDTH, self.KERNEL_SIZE
        )

        # Compute maximum across height and width window dimensions
        MAX_POOLED_TENSOR, _ = RESHAPED_TENSOR.max(dim=3)
        MAX_POOLED_TENSOR, _ = MAX_POOLED_TENSOR.max(dim=4)

        return MAX_POOLED_TENSOR


class SCRATCH_CROSS_ENTROPY_LOSS(torch.nn.Module):
    """Cross-Entropy Loss function from scratch with numerically stable Softmax logic."""

    def forward(self, LOGITS_TENSOR, TARGET_LABELS):
        # Extract maximum logit value per sample for numerical stability adjustment
        MAX_LOGITS, _ = torch.max(LOGITS_TENSOR, dim=1, keepdim=True)
        STABILIZED_LOGITS = LOGITS_TENSOR - MAX_LOGITS

        # Compute Softmax probabilities
        EXPONENTIAL_LOGITS = torch.exp(STABILIZED_LOGITS)
        SOFTMAX_PROBABILITIES = EXPONENTIAL_LOGITS / torch.sum(EXPONENTIAL_LOGITS, dim=1, keepdim=True)

        BATCH_SIZE = LOGITS_TENSOR.size(0)
        BATCH_INDICES = torch.arange(BATCH_SIZE, device=LOGITS_TENSOR.device)

        # Select predicted probabilities corresponding to the target class labels
        CORRECT_CLASS_PROBABILITIES = SOFTMAX_PROBABILITIES[BATCH_INDICES, TARGET_LABELS]

        # Calculate Negative Log Likelihood loss
        COMPUTED_LOSS = -torch.log(CORRECT_CLASS_PROBABILITIES + 1e-9)
        AVERAGE_LOSS = torch.mean(COMPUTED_LOSS)

        return AVERAGE_LOSS


# FULL CNN ARCHITECTURE ASSEMBLY

class FULL_SCRATCH_CNN(torch.nn.Module):
    def __init__(self, NUMBER_OF_CLASSES=10):
        super(FULL_SCRATCH_CNN, self).__init__()

        # Feature Extractor Block 1
        self.CONVOLUTION_LAYER_1 = FROM_SCRATCH_CONVOLUTION_2D(IN_CHANNELS=3, OUT_CHANNELS=16, KERNEL_SIZE=3, PADDING=1)
        self.RELU_ACTIVATION_1 = FROM_SCRATCH_RELU()
        self.MAX_POOLING_LAYER_1 = FROM_SCRATCH_MAX_POOL_2D(KERNEL_SIZE=2, STRIDE=2)

        # Feature Extractor Block 2
        self.CONVOLUTION_LAYER_2 = FROM_SCRATCH_CONVOLUTION_2D(IN_CHANNELS=16, OUT_CHANNELS=32, KERNEL_SIZE=3, PADDING=1)
        self.RELU_ACTIVATION_2 = FROM_SCRATCH_RELU()
        self.MAX_POOLING_LAYER_2 = FROM_SCRATCH_MAX_POOL_2D(KERNEL_SIZE=2, STRIDE=2)

        # Classification Dense Block
        self.FULLY_CONNECTED_LAYER_1 = FROM_SCRATCH_LINEAR(IN_FEATURES=32 * 8 * 8, OUT_FEATURES=128)
        self.RELU_ACTIVATION_3 = FROM_SCRATCH_RELU()
        self.FULLY_CONNECTED_LAYER_2 = FROM_SCRATCH_LINEAR(IN_FEATURES=128, OUT_FEATURES=NUMBER_OF_CLASSES)

    def forward(self, INPUT_TENSOR):
        # Convolutional Block 1
        FEATURE_MAP_1 = self.CONVOLUTION_LAYER_1(INPUT_TENSOR)
        ACTIVATED_MAP_1 = self.RELU_ACTIVATION_1(FEATURE_MAP_1)
        POOLED_MAP_1 = self.MAX_POOLING_LAYER_1(ACTIVATED_MAP_1)

        # Convolutional Block 2
        FEATURE_MAP_2 = self.CONVOLUTION_LAYER_2(POOLED_MAP_1)
        ACTIVATED_MAP_2 = self.RELU_ACTIVATION_2(FEATURE_MAP_2)
        POOLED_MAP_2 = self.MAX_POOLING_LAYER_2(ACTIVATED_MAP_2)

        # Flatten 4D spatial feature tensor into 2D matrix
        FLATTENED_TENSOR = POOLED_MAP_2.view(POOLED_MAP_2.size(0), -1)

        # Fully Connected Classifier Layers
        DENSE_OUTPUT_1 = self.FULLY_CONNECTED_LAYER_1(FLATTENED_TENSOR)
        ACTIVATED_DENSE_1 = self.RELU_ACTIVATION_3(DENSE_OUTPUT_1)
        FINAL_LOGITS_OUTPUT = self.FULLY_CONNECTED_LAYER_2(ACTIVATED_DENSE_1)

        return FINAL_LOGITS_OUTPUT

# TRAINING & EVALUATION PIPELINE

COMPUTE_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"USING COMPUTE DEVICE: {COMPUTE_DEVICE}")

BATCH_SIZE = 64
LEARNING_RATE = 0.001
TOTAL_EPOCHS = 5
NUMBER_OF_CLASSES = 10

# Instantiate custom scratch transforms pipeline
SCRATCH_TRANSFORM_PIPELINE = CUSTOM_COMPOSE_TRANSFORMS([
    CUSTOM_TO_TENSOR(),
    CUSTOM_NORMALIZE(MEAN_TUPLE=(0.4914, 0.4822, 0.4465), STD_TUPLE=(0.2023, 0.1994, 0.2010))
])

# Load CIFAR-10 datasets applying custom transform pipeline
TRAINING_DATASET = torchvision.datasets.CIFAR10(
    root='./DATA', train=True, download=True, transform=SCRATCH_TRANSFORM_PIPELINE
)
TESTING_DATASET = torchvision.datasets.CIFAR10(
    root='./DATA', train=False, download=True, transform=SCRATCH_TRANSFORM_PIPELINE
)

# Instantiate PyTorch DataLoader iterators
TRAIN_DATA_LOADER = torch.utils.data.DataLoader(TRAINING_DATASET, batch_size=BATCH_SIZE, shuffle=True)
TEST_DATA_LOADER = torch.utils.data.DataLoader(TESTING_DATASET, batch_size=BATCH_SIZE, shuffle=False)

# Initialize scratch model, loss function, and optimizer
CNN_MODEL_INSTANCE = FULL_SCRATCH_CNN(NUMBER_OF_CLASSES=NUMBER_OF_CLASSES).to(COMPUTE_DEVICE)
CUSTOM_LOSS_FUNCTION = SCRATCH_CROSS_ENTROPY_LOSS()
OPTIMIZER_ADAM = torch.optim.Adam(CNN_MODEL_INSTANCE.parameters(), lr=LEARNING_RATE)

# TRAINING LOOP
print("\nSTARTING TRAINING LOOP")
for EPOCH_INDEX in range(TOTAL_EPOCHS):
    CNN_MODEL_INSTANCE.train()
    RUNNING_LOSS = 0.0

    for BATCH_INDEX, (BATCH_IMAGES, BATCH_LABELS) in enumerate(TRAIN_DATA_LOADER):
        BATCH_IMAGES = BATCH_IMAGES.to(COMPUTE_DEVICE)
        BATCH_LABELS = BATCH_LABELS.to(COMPUTE_DEVICE)

        OPTIMIZER_ADAM.zero_grad()

        PREDICTION_OUTPUTS = CNN_MODEL_INSTANCE(BATCH_IMAGES)
        CALCULATED_LOSS = CUSTOM_LOSS_FUNCTION(PREDICTION_OUTPUTS, BATCH_LABELS)

        CALCULATED_LOSS.backward()
        OPTIMIZER_ADAM.step()

        RUNNING_LOSS += CALCULATED_LOSS.item()

        if (BATCH_INDEX + 1) % 200 == 0:
            AVERAGE_BATCH_LOSS = RUNNING_LOSS / 200
            print(f"EPOCH [{EPOCH_INDEX + 1}/{TOTAL_EPOCHS}] | BATCH [{BATCH_INDEX + 1}/{len(TRAIN_DATA_LOADER)}] | LOSS: {AVERAGE_BATCH_LOSS:.4f}")
            RUNNING_LOSS = 0.0

# EVALUATION LOOP
print("\nSTARTING EVALUATION")
CNN_MODEL_INSTANCE.eval()
TOTAL_CORRECT_PREDICTIONS = 0
TOTAL_TEST_SAMPLES = 0

with torch.no_grad():
    for BATCH_IMAGES, BATCH_LABELS in TEST_DATA_LOADER:
        BATCH_IMAGES = BATCH_IMAGES.to(COMPUTE_DEVICE)
        BATCH_LABELS = BATCH_LABELS.to(COMPUTE_DEVICE)

        PREDICTION_OUTPUTS = CNN_MODEL_INSTANCE(BATCH_IMAGES)
        _, PREDICTED_CLASSES = torch.max(PREDICTION_OUTPUTS, dim=1)

        TOTAL_TEST_SAMPLES += BATCH_LABELS.size(0)
        TOTAL_CORRECT_PREDICTIONS += (PREDICTED_CLASSES == BATCH_LABELS).sum().item()

FINAL_ACCURACY = (TOTAL_CORRECT_PREDICTIONS / TOTAL_TEST_SAMPLES) * 100.0
print(f"\nFINAL TEST ACCURACY ON CIFAR-10: {FINAL_ACCURACY:.2f}%")

USING COMPUTE DEVICE: cuda


100%|██████████| 170M/170M [34:44<00:00, 81.8kB/s]



STARTING TRAINING LOOP
EPOCH [1/5] | BATCH [200/782] | LOSS: 1.7037
EPOCH [1/5] | BATCH [400/782] | LOSS: 1.3530
EPOCH [1/5] | BATCH [600/782] | LOSS: 1.2542
EPOCH [2/5] | BATCH [200/782] | LOSS: 1.0530
EPOCH [2/5] | BATCH [400/782] | LOSS: 1.0362
EPOCH [2/5] | BATCH [600/782] | LOSS: 1.0142
EPOCH [3/5] | BATCH [200/782] | LOSS: 0.8977
EPOCH [3/5] | BATCH [400/782] | LOSS: 0.9059
EPOCH [3/5] | BATCH [600/782] | LOSS: 0.8942
EPOCH [4/5] | BATCH [200/782] | LOSS: 0.8083
EPOCH [4/5] | BATCH [400/782] | LOSS: 0.8181
EPOCH [4/5] | BATCH [600/782] | LOSS: 0.8279
EPOCH [5/5] | BATCH [200/782] | LOSS: 0.7189
EPOCH [5/5] | BATCH [400/782] | LOSS: 0.7541
EPOCH [5/5] | BATCH [600/782] | LOSS: 0.7498

STARTING EVALUATION

FINAL TEST ACCURACY ON CIFAR-10: 68.02%
